# Libraries

In [4]:
# uncomment this if you dont have tabnet installed
# !pip install pytorch-tabnet

In [62]:
# import libraries
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [6]:
# for google colab
from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Mounted at /content/drive
Using device: cuda


# Read in Files

In [8]:
# load csv files
train_dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datasets/train.csv')
test_dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datasets/test.csv')

# train (cross-validation)
X = train_dataset.drop(columns=["hit"]).to_numpy()
y = train_dataset['hit'].to_numpy()

# test (not used in cross-validation)
X_test = test_dataset.drop(columns=["hit"]).to_numpy()
y_test = test_dataset["hit"].to_numpy()


In [9]:
print("===== TRAINING DATASET =====")
print(f"Full train shape: {train_dataset.shape}")
print(f"X_train shape: {X.shape}")
print(f"y_train shape: {y.shape}")

print("\n===== TEST DATASET =====")
print(f"Full test shape: {test_dataset.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

===== TRAINING DATASET =====
Full train shape: (19348, 31)
X_train shape: (19348, 30)
y_train shape: (19348,)

===== TEST DATASET =====
Full test shape: (4837, 31)
X_test shape: (4837, 30)
y_test shape: (4837,)


# TabNet with Cross Validation

## Train with Cross-Validation

In [38]:
configs = [
    dict(n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=0.001, lr=0.001),
    dict(n_d=32, n_a=32, n_steps=6, gamma=1.5, lambda_sparse=0.001, lr=0.001),
    dict(n_d=32, n_a=32, n_steps=5, gamma=1.8, lambda_sparse=0.01, lr=0.001),
    dict(n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=0,      lr=0.001),
    dict(n_d=64, n_a=64, n_steps=5, gamma=1.5, lambda_sparse=0.001, lr=0.001),
]

# n_d: numner of units in the decision prediction layer

# n_a: number of units in the attention layer
# (how the model selects important features at each step)

# n_steps: number of sequential steps,
# (how many times the model applies attention + decision blocks)

# gamma: controls how many features can be reused across steps

# lambda_sparse: Encourages the model to use fewer features at each step.

# lr: step size used by the optimizer when updating weights


skf = StratifiedKFold(n_splits=5, random_state=426, shuffle=True)
results = []

for ci, cfg in enumerate(configs, 1):
    fold_aucs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        # split
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # model from config
        tb_cls = TabNetClassifier(
            n_d=cfg["n_d"],
            n_a=cfg["n_a"],
            n_steps=cfg["n_steps"],
            gamma=cfg["gamma"],
            lambda_sparse=cfg["lambda_sparse"],
            optimizer_fn=torch.optim.AdamW,
            optimizer_params=dict(lr=cfg["lr"], weight_decay=1e-5),
            mask_type="entmax",
            device_name=device
        )

        # train
        tb_cls.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_name=['val'],
            eval_metric=["auc"],
            max_epochs=100,
            patience=10,
            batch_size=1024,
            virtual_batch_size=128,
            drop_last=False
        )

        # AUC from probabilities
        val_proba = tb_cls.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, val_proba)
        fold_aucs.append(auc)

        print(f"Config {ci}/{len(configs)} | Fold {fold} AUC: {auc:.4f}")

    mean_auc = float(np.mean(fold_aucs))
    std_auc  = float(np.std(fold_aucs))

    results.append((ci, cfg, mean_auc, std_auc))

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.67017 | val_auc: 0.48718 |  0:00:00s
epoch 1  | loss: 0.56115 | val_auc: 0.49323 |  0:00:00s
epoch 2  | loss: 0.52265 | val_auc: 0.48477 |  0:00:01s
epoch 3  | loss: 0.50355 | val_auc: 0.5032  |  0:00:01s
epoch 4  | loss: 0.48772 | val_auc: 0.50643 |  0:00:01s
epoch 5  | loss: 0.47809 | val_auc: 0.52686 |  0:00:02s
epoch 6  | loss: 0.46246 | val_auc: 0.53695 |  0:00:02s
epoch 7  | loss: 0.45326 | val_auc: 0.56738 |  0:00:02s
epoch 8  | loss: 0.44707 | val_auc: 0.58186 |  0:00:03s
epoch 9  | loss: 0.43646 | val_auc: 0.59923 |  0:00:03s
epoch 10 | loss: 0.43471 | val_auc: 0.60065 |  0:00:04s
epoch 11 | loss: 0.42872 | val_auc: 0.61584 |  0:00:04s
epoch 12 | loss: 0.42493 | val_auc: 0.6224  |  0:00:04s
epoch 13 | loss: 0.42203 | val_auc: 0.63172 |  0:00:05s
epoch 14 | loss: 0.4206  | val_auc: 0.63652 |  0:00:05s
epoch 15 | loss: 0.41746 | val_auc: 0.64944 |  0:00:05s
epoch 16 | loss: 0.42012 | val_auc: 0.65831 |  0:00:06s
epoch 17 | loss: 0.41308 | val_auc: 0.65066 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.66281 | val_auc: 0.49306 |  0:00:00s
epoch 1  | loss: 0.57578 | val_auc: 0.48933 |  0:00:00s
epoch 2  | loss: 0.53071 | val_auc: 0.50055 |  0:00:01s
epoch 3  | loss: 0.49772 | val_auc: 0.50534 |  0:00:01s
epoch 4  | loss: 0.49063 | val_auc: 0.50975 |  0:00:01s
epoch 5  | loss: 0.46425 | val_auc: 0.53435 |  0:00:02s
epoch 6  | loss: 0.46147 | val_auc: 0.54597 |  0:00:02s
epoch 7  | loss: 0.45249 | val_auc: 0.56084 |  0:00:02s
epoch 8  | loss: 0.44361 | val_auc: 0.57048 |  0:00:03s
epoch 9  | loss: 0.44005 | val_auc: 0.58253 |  0:00:03s
epoch 10 | loss: 0.43146 | val_auc: 0.59419 |  0:00:03s
epoch 11 | loss: 0.4275  | val_auc: 0.59456 |  0:00:04s
epoch 12 | loss: 0.43052 | val_auc: 0.60413 |  0:00:04s
epoch 13 | loss: 0.42349 | val_auc: 0.62065 |  0:00:05s
epoch 14 | loss: 0.41932 | val_auc: 0.63407 |  0:00:05s
epoch 15 | loss: 0.41824 | val_auc: 0.63233 |  0:00:05s
epoch 16 | loss: 0.41224 | val_auc: 0.62534 |  0:00:06s
epoch 17 | loss: 0.41221 | val_auc: 0.63262 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


...

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.68704 | val_auc: 0.46124 |  0:00:00s
epoch 1  | loss: 0.57415 | val_auc: 0.49837 |  0:00:00s
epoch 2  | loss: 0.5246  | val_auc: 0.49759 |  0:00:01s
epoch 3  | loss: 0.4814  | val_auc: 0.49869 |  0:00:01s
epoch 4  | loss: 0.45771 | val_auc: 0.51968 |  0:00:01s
epoch 5  | loss: 0.45434 | val_auc: 0.54178 |  0:00:02s
epoch 6  | loss: 0.44498 | val_auc: 0.54647 |  0:00:02s
epoch 7  | loss: 0.44504 | val_auc: 0.55632 |  0:00:02s
epoch 8  | loss: 0.43002 | val_auc: 0.54732 |  0:00:03s
epoch 9  | loss: 0.42686 | val_auc: 0.55683 |  0:00:03s
epoch 10 | loss: 0.424   | val_auc: 0.57313 |  0:00:03s
epoch 11 | loss: 0.42164 | val_auc: 0.57888 |  0:00:04s
epoch 12 | loss: 0.42717 | val_auc: 0.58678 |  0:00:04s
epoch 13 | loss: 0.41916 | val_auc: 0.5885  |  0:00:04s
epoch 14 | loss: 0.41545 | val_auc: 0.5969  |  0:00:05s
epoch 15 | loss: 0.41254 | val_auc: 0.60095 |  0:00:05s
epoch 16 | loss: 0.40977 | val_auc: 0.61727 |  0:00:05s
epoch 17 | loss: 0.41403 | val_auc: 0.61521 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.66446 | val_auc: 0.47572 |  0:00:00s
epoch 1  | loss: 0.58453 | val_auc: 0.5261  |  0:00:00s
epoch 2  | loss: 0.52168 | val_auc: 0.51506 |  0:00:01s
epoch 3  | loss: 0.49088 | val_auc: 0.52245 |  0:00:01s
epoch 4  | loss: 0.47655 | val_auc: 0.55146 |  0:00:01s
epoch 5  | loss: 0.45511 | val_auc: 0.56717 |  0:00:02s
epoch 6  | loss: 0.45085 | val_auc: 0.56885 |  0:00:02s
epoch 7  | loss: 0.44227 | val_auc: 0.57791 |  0:00:02s
epoch 8  | loss: 0.43544 | val_auc: 0.57805 |  0:00:03s
epoch 9  | loss: 0.43173 | val_auc: 0.59248 |  0:00:03s
epoch 10 | loss: 0.43545 | val_auc: 0.60716 |  0:00:03s
epoch 11 | loss: 0.42593 | val_auc: 0.61117 |  0:00:04s
epoch 12 | loss: 0.42431 | val_auc: 0.62241 |  0:00:04s
epoch 13 | loss: 0.42687 | val_auc: 0.62145 |  0:00:04s
epoch 14 | loss: 0.41874 | val_auc: 0.61295 |  0:00:05s
epoch 15 | loss: 0.41389 | val_auc: 0.6404  |  0:00:05s
epoch 16 | loss: 0.41466 | val_auc: 0.6258  |  0:00:06s
epoch 17 | loss: 0.41713 | val_auc: 0.62995 |  0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


## Summary of Results from CV

In [39]:
# sort best to worst
results_sorted = sorted(results, key=lambda x: x[2], reverse=True)

print("===== Summary (best first) =====")
for ci, cfg, mean_auc, std_auc in results_sorted:
    print(f"Config {ci}: AUC {mean_auc:.4f} ± {std_auc:.4f} | {cfg}")

===== Summary (best first) =====
Config 3: AUC 0.6702 ± 0.0094 | {'n_d': 32, 'n_a': 32, 'n_steps': 5, 'gamma': 1.8, 'lambda_sparse': 0.01, 'lr': 0.001}
Config 4: AUC 0.6692 ± 0.0080 | {'n_d': 32, 'n_a': 32, 'n_steps': 5, 'gamma': 1.5, 'lambda_sparse': 0, 'lr': 0.001}
Config 1: AUC 0.6653 ± 0.0048 | {'n_d': 32, 'n_a': 32, 'n_steps': 5, 'gamma': 1.5, 'lambda_sparse': 0.001, 'lr': 0.001}
Config 2: AUC 0.6653 ± 0.0131 | {'n_d': 32, 'n_a': 32, 'n_steps': 6, 'gamma': 1.5, 'lambda_sparse': 0.001, 'lr': 0.001}
Config 5: AUC 0.6642 ± 0.0116 | {'n_d': 64, 'n_a': 64, 'n_steps': 5, 'gamma': 1.5, 'lambda_sparse': 0.001, 'lr': 0.001}


## Pick Best Config from Cross Validation Results

In [42]:
best_ci, best_cfg, best_mean_auc, best_std = max(results, key=lambda x: x[2])
print(f"Best config = {best_ci} | CV AUC = {best_mean_auc:.4f} ± {best_std:.4f}")
print(best_cfg)

Best config = 3 | CV AUC = 0.6702 ± 0.0094
{'n_d': 32, 'n_a': 32, 'n_steps': 5, 'gamma': 1.8, 'lambda_sparse': 0.01, 'lr': 0.001}


## Train Final Model on All Training Data

In [44]:
# split the dataset into test and val again but doing it on a smaller val

# the validation set here simply determines when to stop training
# to reduce overfitting during final model training.
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.1, random_state=426, stratify=y
)

# build findal model using best hyperparameters selected from CV
final_model = TabNetClassifier(
    n_d=best_cfg["n_d"],
    n_a=best_cfg["n_a"],
    n_steps=best_cfg["n_steps"],
    gamma=best_cfg["gamma"],
    lambda_sparse=best_cfg["lambda_sparse"],
    optimizer_fn=torch.optim.AdamW,
    optimizer_params=dict(lr=best_cfg["lr"], weight_decay=1e-5),
    mask_type="entmax",
    device_name=device
)

# train final model with early stopping
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric=["auc"],
    max_epochs=100,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128,
    drop_last=False
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.67866 | val_0_auc: 0.47299 |  0:00:00s
epoch 1  | loss: 0.57781 | val_0_auc: 0.47986 |  0:00:00s
epoch 2  | loss: 0.54047 | val_0_auc: 0.47548 |  0:00:01s
epoch 3  | loss: 0.504   | val_0_auc: 0.47056 |  0:00:01s
epoch 4  | loss: 0.48458 | val_0_auc: 0.49773 |  0:00:01s
epoch 5  | loss: 0.47658 | val_0_auc: 0.53323 |  0:00:02s
epoch 6  | loss: 0.46634 | val_0_auc: 0.54545 |  0:00:02s
epoch 7  | loss: 0.45431 | val_0_auc: 0.55454 |  0:00:03s
epoch 8  | loss: 0.45261 | val_0_auc: 0.5545  |  0:00:03s
epoch 9  | loss: 0.44459 | val_0_auc: 0.56165 |  0:00:03s
epoch 10 | loss: 0.4335  | val_0_auc: 0.58838 |  0:00:04s
epoch 11 | loss: 0.43596 | val_0_auc: 0.60983 |  0:00:04s
epoch 12 | loss: 0.43003 | val_0_auc: 0.62307 |  0:00:05s
epoch 13 | loss: 0.42734 | val_0_auc: 0.62005 |  0:00:05s
epoch 14 | loss: 0.42605 | val_0_auc: 0.62478 |  0:00:05s
epoch 15 | loss: 0.42501 | val_0_auc: 0.62593 |  0:00:06s
epoch 16 | loss: 0.42015 | val_0_auc: 0.62086 |  0:00:06s
epoch 17 | los

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


## Test Set Results:

In [49]:
test_proba = final_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_proba)
print(f"TEST AUC: {test_auc:.4f}")

TEST AUC: 0.6489


## Confustion Matrix:

In [53]:
# Predict class labels
y_pred = final_model.predict(X_val)

# Compute confusion matrix
cm = confusion_matrix(y_val, y_pred)

print(cm)

[[1656    2]
 [ 275    2]]


In [60]:
y_proba = final_model.predict_proba(X_val)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("best threshold:", best_threshold)
print("best f1:", f1_scores[best_idx])

best threshold: 0.19179867
best f1: 0.35851851367918797


In [61]:
y_pred = (y_proba >= best_threshold).astype(int)
cm = confusion_matrix(y_val, y_pred)

print(cm)

[[1381  277]
 [ 156  121]]


In [63]:
pr_auc = average_precision_score(y_val, y_proba)
print("PR AUC:", pr_auc)

PR AUC: 0.28090036114472244


## Interpretation:

We can’t just look at the raw confusion matrix because the dataset is imbalanced (only 14.3% hits), which means a model can appear “good” simply by predicting mostly non-hits. At the default 0.5 threshold, the confusion matrix shows very few positive predictions (only 2 true positives), giving extremely low recall despite high overall accuracy — this is misleading because the model is essentially ignoring the minority class. After adjusting the threshold to maximize F1 (~0.19), the model captures about 44% of hits with ~30% precision, reflecting a much more meaningful trade-off. More importantly, the PR AUC of 0.281 is nearly double the baseline of 0.143 (the class prevalence), indicating that the model has genuine ranking power beyond random guessing. Overall, the model does learn real signal, but proper threshold tuning is essential to obtain practically useful performance in this imbalanced setting.